# Module 5: Clipping Defense


## Stage Goal

This notebook isolates norm-clipping aggregation. It loads the saved FedAvg baselines from `fedavg_baselines.ipynb`, runs one attacked clipping defense, and compares it against clean and attacked FedAvg. Run `fedavg_baselines.ipynb` first so the clean and attacked FedAvg artifacts exist. Sweeps stay in `defense_sweeps.ipynb`.


## 1. Notebook Setup


In [1]:
from pathlib import Path
import sys

MODULE_DIR = Path.cwd()
if (MODULE_DIR / "5_Defensive_FL").exists():
    MODULE_DIR = MODULE_DIR / "5_Defensive_FL"
MODULE4_DIR = MODULE_DIR.parent / "4_Adversarial_FL"
MODULE4_SRC_DIR = MODULE4_DIR / "src"
for path in (MODULE_DIR.parent, MODULE4_DIR, MODULE4_SRC_DIR, MODULE_DIR):
    path_text = str(path.resolve())
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from IPython.display import Image, display
import pandas as pd

from src.notebook_utils import (
    load_required_baselines,
    prepare_context,
    record_config_snapshot,
    run_module5_experiment,
    run_result_path,
    validate_artifacts,
    validate_result_collection,
)
from src.metrics import build_comparison_rows, latest_defense_diagnostics
from src.plots import (
    plot_accuracy_curves,
    plot_defense_comparison,
    plot_global_target_label_asr_curves,
    plot_surrogate_poison_success_curves,
)


## 2. Configuration

Edit this cell to change data, FL, attack, or defense settings. No YAML config is used.


In [2]:
BASE_FED_CONFIG = {
    "algorithm": "FedAvg",
    "fraction_clients": 0.2,
    "num_clients": 100,
    "num_rounds": 30,
    "num_epochs": 5,
    "batch_size": 64,
    "global_stepsize": 1.0,
    "local_stepsize": 0.002,
    "criterion": "torch.nn.CrossEntropyLoss",
}

RUN_NAME = 'clipping'
DEFENSE_TITLE = 'Norm clipping'
DEFENSE_CONFIG = {'name': 'clipping', 'clip_norm': 5.0}

CONFIG = {
    "data_config": {
        "dataset_path": "./Data/Imagenette",
        "dataset_name": "Imagenette",
        "non_iid_per": 0,
        "eval_subset": "attack_eval",
        "validation_split": {"enabled": True, "seed": 42, "selection_fraction": 0.5},
    },
    "global_config": {"seed": 42, "device": "cuda"},
    "module4_handoff": {
        "enabled": True,
        "artifacts_dir": "../4_Adversarial_FL/artifacts",
        "target_checkpoint": "module4_v3_target.pt",
        "surrogate_checkpoint": "module4_surrogate.pt",
    },
    "artifacts": {
        "dir": "artifacts",
        "config_snapshot": f"module5_{RUN_NAME}_config_used.json",
        "clean_baseline_metrics": "module5_clean_fedavg.json",
        "attacked_baseline_metrics": "module5_attacked_fedavg.json",
    },
    "stage": {"name": f"{RUN_NAME}_defense", "notebook": 'clipping_defense.ipynb'},
    "model_config": {
        "module": "model",
        "name": "MobileNetV3Transfer",
        "kwargs": {"pretrained": False, "num_classes": 10, "dropout": 0.1},
    },
    "algorithms": {
        "FedAvg": {
            "fed_config": BASE_FED_CONFIG,
            "optim_config": {},
        }
    },
    "defense": DEFENSE_CONFIG,
    "attack": {
        "seed": 42,
        "malicious_fraction": 0.2,
        "malicious_client_selection": {"mode": "seeded_random", "client_ids": []},
        "start_round": 1,
        "attack": {
            "type": "pgd",
            "poison_rate": 0.4,
            "target_label": 0,
            "epsilon": 8 / 255,
            "step_size": 2 / 255,
            "iters": 30,
            "criterion": "torch.nn.CrossEntropyLoss",
            "poison_rate_schedule": None,
            # "poison_rate_schedule": {"type": "linear", "start": 0.05, "end": 0.2},
        },
        "surrogate": {
            "checkpoint": "../4_Adversarial_FL/artifacts/module4_surrogate.pt",
            "checkpoint_source": "train_surrogate.ipynb",
            "pretrained": False,
            "finetune_epochs": 0,
            "local_finetune_epochs": 0,
            "learning_rate": 0.001,
            "weight_decay": 0.0,
            "batch_size": 64,
            "freeze_backbone": False,
            "early_stop_patience": 0,
        },
    },
}

context = prepare_context(CONFIG, stage_name=f"{RUN_NAME}_defense")
config_snapshot_path = record_config_snapshot(context)
print(f"Defense: {DEFENSE_TITLE}")
print(f"Sampled clients per round: {context.sampled_clients}")
print(f"Config snapshot: {config_snapshot_path}")


Loaded notebook cell: stage=clipping_defense, rounds=30, sampled_clients=20, eval_subset=attack_eval.
Defense: Norm clipping
Sampled clients per round: 20
Config snapshot: /home/ahoop004/T3-Ciders-FL/5_Defensive_FL/artifacts/module5_clipping_config_used.json


## 3. Load FedAvg Baseline Artifacts

This loads the clean and attacked FedAvg results saved by `fedavg_baselines.ipynb`; it does not rerun those baselines.


In [3]:
baseline_results = load_required_baselines(context)
baseline_validation = validate_result_collection(
    context,
    baseline_results,
    required_runs=["clean_fedavg", "attacked_fedavg"],
)

display(pd.DataFrame(baseline_validation))


,run,rounds,final_accuracy,final_surrogate_poison_success_rate,final_global_target_label_asr
0,clean_fedavg,30,97.757390,0.0,0.226116
1,attacked_fedavg,30,96.279307,100.0,1.074053


## 4. Run Norm clipping

This runs one attacked FL job with only `DEFENSE_CONFIG` changed from attacked FedAvg.


In [ ]:
defense_result = run_module5_experiment(
    context,
    RUN_NAME,
    context.base_attack_config,
    DEFENSE_CONFIG,
)
run_results = {**baseline_results, RUN_NAME: defense_result}
comparison_rows = build_comparison_rows(run_results)
comparison_table = pd.DataFrame(comparison_rows)

display(comparison_table)
display(pd.DataFrame(validate_result_collection(context, {RUN_NAME: defense_result})))
latest_defense_diagnostics(defense_result)


Defensive server initialized



Preparing data with Dirichlet partitioner (aligned with Module 2)
Loading cached client data from cache/client_data_7aefb7fbe94af4a610cf3e5bf7789b72.pkl


Clients successfully initialised
Initialized defensive server from /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_v3_target.pt

Communication Round: 1


[ClippingServer] Round 1/30 selected 20 clients, 5 local epoch(s), defense=clipping


	client_update completed
	server_update completed
	Loss: 0.0759   Accuracy: 97.71%

Communication Round: 2


	Server Loss: 0.0759   Accuracy: 97.71%
[ClippingServer] Round 2/30 selected 20 clients, 5 local epoch(s), defense=clipping


## 5. Plots and Artifact Validation

The plots compare clean FedAvg, attacked FedAvg, and this defense notebook's run.


In [ ]:
plot_paths = [
    plot_accuracy_curves(
        run_results,
        context.artifact_path(f"module5_{RUN_NAME}_accuracy_curves.png"),
        attack_start_round=context.base_attack_config["start_round"],
        title=f"{DEFENSE_TITLE}: accuracy by round",
    ),
    plot_surrogate_poison_success_curves(
        run_results,
        context.artifact_path(f"module5_{RUN_NAME}_surrogate_poison_success_curves.png"),
        attack_start_round=context.base_attack_config["start_round"],
        title=f"{DEFENSE_TITLE}: surrogate poison success by round",
    ),
    plot_global_target_label_asr_curves(
        run_results,
        context.artifact_path(f"module5_{RUN_NAME}_global_target_label_asr_curves.png"),
        attack_start_round=context.base_attack_config["start_round"],
        title=f"{DEFENSE_TITLE}: global target-label ASR by round",
    ),
    plot_defense_comparison(
        comparison_rows,
        metric="final_accuracy",
        path=context.artifact_path(f"module5_{RUN_NAME}_accuracy_vs_baselines.png"),
        ylabel="Final accuracy (%)",
        title=f"{DEFENSE_TITLE}: final accuracy",
    ),
    plot_defense_comparison(
        comparison_rows,
        metric="final_surrogate_poison_success_rate",
        path=context.artifact_path(f"module5_{RUN_NAME}_surrogate_poison_success_vs_baselines.png"),
        ylabel="Final surrogate poison success rate (%)",
        title=f"{DEFENSE_TITLE}: final surrogate poison success",
    ),
    plot_defense_comparison(
        comparison_rows,
        metric="final_global_target_label_asr",
        path=context.artifact_path(f"module5_{RUN_NAME}_global_target_label_asr_vs_baselines.png"),
        ylabel="Final global target-label ASR (%)",
        title=f"{DEFENSE_TITLE}: final global target-label ASR",
    ),
]

validate_artifacts([config_snapshot_path, run_result_path(context, RUN_NAME), *plot_paths])
for plot_path in plot_paths:
    display(Image(filename=str(plot_path)))
